In [13]:
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

# VOXEL_SIZE = (4, 1, 1)  # Z, Y, X
# VOXEL_VOLUME = np.prod(VOXEL_SIZE)



VIDEOS = [
    "44b6_33b596bf", "44b6_40c45f5a", "44b6_551a5dba", # revised
    
    "44b6_87bba6c4", 
    "44b6_abf82518", 
    "44b6_c96cfa10",

    "44b6_eb2880fc", "6bba_1f58c2f6", "6bba_55c70843",
    "6bba_61dd1e0d", "6bba_6ca87370", "6bba_6feeb0b1",
    "6bba_80d12824", "6bba_9e23430b", "6bba_debd7bfa"
]


MASK_DIR = "deadends/stardist/cellpose_mask/cellpose_cache"


In [18]:
all_radii = []
video_stats = []

for vid in VIDEOS:
    mask = np.load(f"{MASK_DIR}/{vid}_masks.npy", mmap_mode="r")
    video_radii = []

    # Mask shape: (T, Z, Y, X)
    for t in range(mask.shape[0]):
        frame = mask[t]
        objects = ndi.find_objects(frame)

        for bbox in objects:
            if bbox is None:
                continue

            # Raw voxel-coordinate bounding-box half-extents
            rz = (bbox[0].stop - bbox[0].start) / 2
            ry = (bbox[1].stop - bbox[1].start) / 2
            rx = (bbox[2].stop - bbox[2].start) / 2

            video_radii.append((rz, ry, rx))

    video_radii = np.asarray(video_radii, dtype=np.float64).reshape(-1, 3)
    all_radii.extend(video_radii.tolist())

    stats = {
        "video": vid,
        "instances": len(video_radii),
    }

    for i, axis in enumerate(["z", "y", "x"]):
        stats[f"min_r{axis}"] = video_radii[:, i].min() if len(video_radii) else np.nan
        stats[f"median_r{axis}"] = np.median(video_radii[:, i]) if len(video_radii) else np.nan
        stats[f"max_r{axis}"] = video_radii[:, i].max() if len(video_radii) else np.nan

    video_stats.append(stats)
    print(f"Processed {vid}: {len(video_radii)} instances")

Processed 44b6_33b596bf: 19866 instances
Processed 44b6_40c45f5a: 33875 instances
Processed 44b6_551a5dba: 63896 instances
Processed 44b6_87bba6c4: 24439 instances
Processed 44b6_abf82518: 22969 instances
Processed 44b6_c96cfa10: 46957 instances
Processed 44b6_eb2880fc: 35693 instances
Processed 6bba_1f58c2f6: 10421 instances
Processed 6bba_55c70843: 26481 instances
Processed 6bba_61dd1e0d: 8378 instances
Processed 6bba_6ca87370: 6291 instances
Processed 6bba_6feeb0b1: 6607 instances
Processed 6bba_80d12824: 7808 instances
Processed 6bba_9e23430b: 6411 instances
Processed 6bba_debd7bfa: 9855 instances


In [20]:
video_df = pd.DataFrame(video_stats)
display(video_df.round(3))

# Pooled statistics across all videos
all_radii = np.asarray(all_radii, dtype=np.float64).reshape(-1, 3)

dataset_stats = {"video": "ALL VIDEOS", "instances": len(all_radii)}

for i, axis in enumerate(["z", "y", "x"]):
    dataset_stats[f"min_r{axis}"] = all_radii[:, i].min() if len(all_radii) else np.nan
    dataset_stats[f"median_r{axis}"] = np.median(all_radii[:, i]) if len(all_radii) else np.nan
    dataset_stats[f"max_r{axis}"] = all_radii[:, i].max() if len(all_radii) else np.nan

dataset_df = pd.DataFrame([dataset_stats])
display(dataset_df.round(3))

,video,instances,min_rz,median_rz,max_rz,min_ry,median_ry,max_ry,min_rx,median_rx,max_rx
0,44b6_33b596bf,19866,0.5,4.0,18.0,1.0,10.5,34.5,1.5,10.5,29.5
1,44b6_40c45f5a,33875,0.5,3.5,26.0,1.5,11.0,45.5,2.0,10.5,46.0
2,44b6_551a5dba,63896,0.5,3.5,27.5,1.5,10.0,38.0,1.5,11.5,42.5
3,44b6_87bba6c4,24439,0.5,4.0,28.0,2.0,11.5,49.5,2.0,11.5,57.0
4,44b6_abf82518,22969,0.5,2.5,31.0,2.0,12.5,68.0,1.5,12.0,62.0
5,44b6_c96cfa10,46957,0.5,3.5,19.5,1.5,9.5,40.0,1.5,10.0,37.5
6,44b6_eb2880fc,35693,0.5,2.0,30.0,1.5,12.0,99.5,2.0,12.0,102.5
7,6bba_1f58c2f6,10421,0.5,1.0,22.0,2.0,10.0,111.0,2.0,10.0,112.0
8,6bba_55c70843,26481,0.5,1.5,13.5,1.5,9.0,96.5,1.5,8.5,97.5
9,6bba_61dd1e0d,8378,0.5,4.5,25.0,2.0,12.0,75.0,2.0,11.5,75.0


,video,instances,min_rz,median_rz,max_rz,min_ry,median_ry,max_ry,min_rx,median_rx,max_rx
0,ALL VIDEOS,329947,0.5,3.0,31.0,1.0,10.5,111.0,1.0,11.0,112.0
